In [1]:
# ─────────────────────────── standard libs ────────────────────────────────
import math
import warnings
warnings.filterwarnings('ignore')

# ────────────────────────────── data stack ────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ────────────────────────────── sklearn ───────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix,
)

# ─────────────────────────────── torch ────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler, random_split

from tqdm import tqdm

print('All imports OK')

All imports OK


## 1. Dataset Overview

In [2]:
# ─── Load CSV ─────────────────────────────────────────────────────────────
CSV_PATH = '/kaggle/input/datasets/harunshimanto/epileptic-seizure-recognition/Epileptic Seizure Recognition.csv'

df_raw = pd.read_csv(CSV_PATH)
print(f'Raw CSV shape : {df_raw.shape}')   # (11500, 180)
print(df_raw.head(3))

Raw CSV shape : (11500, 180)
      Unnamed   X1   X2   X3   X4   X5   X6   X7   X8   X9  ...  X170  X171  \
0  X21.V1.791  135  190  229  223  192  125   55   -9  -33  ...   -17   -15   
1  X15.V1.924  386  382  356  331  320  315  307  272  244  ...   164   150   
2     X8.V1.1  -32  -39  -47  -37  -32  -36  -57  -73  -85  ...    57    64   

   X172  X173  X174  X175  X176  X177  X178  y  
0   -31   -77  -103  -127  -116   -83   -51  4  
1   146   152   157   156   154   143   129  1  
2    48    19   -12   -30   -35   -35   -36  5  

[3 rows x 180 columns]


In [3]:
# ─── Extract features and labels ──────────────────────────────────────────
# Columns: 0=row_name, 1..178=features, 179=label
# We use features at columns index 1:178 → 177 time-step values
# Label is at column index 179

values = df_raw.values  # numpy array, shape (11500, 180)

esrinput = values[0:, 1:178].astype(np.float32)   # (11500, 177)
esrlabel = values[0:, 179].astype(np.float32)      # (11500,)

print(f'Input shape : {esrinput.shape}')   # (11500, 177)
print(f'Label shape : {esrlabel.shape}')   # (11500,)

# ─── Binarise labels: 1 = seizure, 2/3/4/5 = non-seizure → 0 ─────────────
esrlabel_binary = (esrlabel == 1).astype(np.int64)  # 1 if seizure, else 0

print(f'\nClass distribution (original):')
for cls in sorted(np.unique(esrlabel)):
    print(f'  Class {int(cls)} : {(esrlabel == cls).sum():,} samples')

print(f'\nBinary labels:')
print(f'  Seizure     (1) : {esrlabel_binary.sum():,}')
print(f'  Non-seizure (0) : {(esrlabel_binary == 0).sum():,}')

Input shape : (11500, 177)
Label shape : (11500,)

Class distribution (original):
  Class 1 : 2,300 samples
  Class 2 : 2,300 samples
  Class 3 : 2,300 samples
  Class 4 : 2,300 samples
  Class 5 : 2,300 samples

Binary labels:
  Seizure     (1) : 2,300
  Non-seizure (0) : 9,200


## 2. Preprocessing

Since this CSV is already a pre-extracted feature set (no raw EDF files),  
preprocessing is limited to:
1. **StandardScaler** normalisation across all 177 features
2. **Reshape** to `(N, 1, 177)` — treating the 177 values as a single-channel time series

No bandpass filtering or resampling is needed (data is already processed).

In [4]:
# ─── StandardScaler normalisation ─────────────────────────────────────────
sc = StandardScaler()
esrinput_scaled = sc.fit_transform(esrinput)   # (11500, 177)

# ─── Reshape to (N, 1, 177): (batch, channels, time_steps) ────────────────
# We treat the 177-feature vector as a single-channel EEG segment.
X = torch.tensor(esrinput_scaled, dtype=torch.float32).unsqueeze(1)  # (11500, 1, 177)
y = torch.tensor(esrlabel_binary, dtype=torch.long)                   # (11500,)

print(f'X shape : {X.shape}')   # torch.Size([11500, 1, 177])
print(f'y shape : {y.shape}')   # torch.Size([11500])

X shape : torch.Size([11500, 1, 177])
y shape : torch.Size([11500])


## 3. Train / Val / Test Split

In [5]:
# ─── 60/20/20 split ───────────────────────────────────────────────────────
full_dataset = TensorDataset(X, y)
total_size   = len(full_dataset)

train_size = int(0.60 * total_size)
eval_size  = int(0.20 * total_size)
test_size  = total_size - train_size - eval_size

torch.manual_seed(42)
train_dataset, eval_dataset, test_dataset = random_split(
    full_dataset, [train_size, eval_size, test_size]
)

print(f'Train : {len(train_dataset):,}')
print(f'Val   : {len(eval_dataset):,}')
print(f'Test  : {len(test_dataset):,}')

# ─── DataLoaders with WeightedRandomSampler for training ──────────────────
BATCH_SIZE = 128

# Build per-sample weights for the training subset
train_labels = y[train_dataset.indices].numpy()
counts       = np.bincount(train_labels)
weights      = 1.0 / counts[train_labels]
sampler      = WeightedRandomSampler(
    torch.from_numpy(weights).float(),
    num_samples=len(weights), replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader   = DataLoader(eval_dataset,  batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

# Sanity check batch shape
xb, yb = next(iter(train_loader))
print(f'\nTrain batch X : {xb.shape}')   # torch.Size([128, 1, 177])
print(f'Train batch y : {yb.shape}')   # torch.Size([128])
print(f'Class balance : seizure={yb.sum().item()}  non={BATCH_SIZE - yb.sum().item()}')

Train : 6,900
Val   : 2,300
Test  : 2,300

Train batch X : torch.Size([128, 1, 177])
Train batch y : torch.Size([128])
Class balance : seizure=81  non=47


## 4. Model — L_SCLNet with EEGformer Backbone

Configured for:
- `input_channels = 1` (single channel)
- `time_steps = 177` (177-point feature vector treated as time series)
- `kernel_size = 5` — reduced from 10 because 177 is much shorter than 512;
  with kernel=5 the ODCM output is S = 177 − 3×(5−1) = **165** time steps.
- `num_submatrices = 6`, `num_heads_ttm = 6` (D=120 divisible by 6)

In [6]:
# ─────────────────────────────────────────────────────────────────────────
# EEGformer components
# ─────────────────────────────────────────────────────────────────────────

def trunc_normal(tensor, mean=0., std=1., a=-2., b=2.):
    def norm_cdf(x):
        return (1. + math.erf(x / math.sqrt(2.))) / 2.
    with torch.no_grad():
        l = norm_cdf((a - mean) / std)
        u = norm_cdf((b - mean) / std)
        tensor.uniform_(2 * l - 1, 2 * u - 1)
        tensor.erfinv_()
        tensor.mul_(std * math.sqrt(2.))
        tensor.add_(mean)
        tensor.clamp_(min=a, max=b)
        return tensor


class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None,
                 act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features    = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1  = nn.Linear(in_features, hidden_features)
        self.act  = act_layer()
        self.fc2  = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))


class GenericTFB(nn.Module):
    def __init__(self, emb_size, num_heads):
        super().__init__()
        self.M  = emb_size
        self.hA = num_heads
        self.Dh = emb_size // num_heads
        self.Wqkv = nn.Parameter(torch.randn(3, num_heads, self.Dh, emb_size))
        self.Wo   = nn.Parameter(torch.randn(emb_size, emb_size))
        self.ln1  = nn.LayerNorm(emb_size)
        self.ln2  = nn.LayerNorm(emb_size)
        self.mlp  = Mlp(emb_size, emb_size * 4)

    def forward(self, x, z):
        qkv = torch.einsum('xhdm,ijm->xijhd', self.Wqkv, self.ln1(z))
        q, k, v = qkv[0], qkv[1], qkv[2]
        att = (q.transpose(1, 2) / math.sqrt(self.Dh)) @ k.transpose(1, 2).transpose(-2, -1)
        att = torch.softmax(att, dim=-1)
        imv = (att @ v.transpose(1, 2)).transpose(1, 2)
        z = torch.einsum('nm,ijm->ijn', self.Wo,
                         imv.reshape(z.shape[0], z.shape[1], self.M)) + z
        z = self.mlp(self.ln2(z)) + z
        return z


class TemporalTFB(nn.Module):
    def __init__(self, emb_size, num_heads, avgf):
        super().__init__()
        self.M    = emb_size
        self.hA   = num_heads
        self.Dh   = emb_size // num_heads
        self.avgf = avgf
        self.Wqkv = nn.Parameter(torch.randn(3, num_heads, self.Dh, emb_size))
        self.Wo   = nn.Parameter(torch.randn(emb_size, emb_size))
        self.ln1  = nn.LayerNorm(emb_size)
        self.ln2  = nn.LayerNorm(emb_size)
        self.mlp  = Mlp(emb_size, emb_size * 4)

    def forward(self, x, z):
        qkv = torch.einsum('xhdm,im->xihd', self.Wqkv, self.ln1(z))
        q, k, v = qkv[0], qkv[1], qkv[2]
        att = (q.transpose(0, 1) / math.sqrt(self.Dh)) @ k.transpose(0, 1).transpose(-2, -1)
        att = torch.softmax(att, dim=-1)
        imv = (att @ v.transpose(0, 1)).transpose(0, 1)
        z = torch.einsum('nm,im->in', self.Wo,
                         imv.reshape(self.avgf + 1, self.M)) + z
        z = self.mlp(self.ln2(z)) + z
        return z


class ODCM(nn.Module):
    """
    Three-layer depthwise-conv feature extractor.
    Input  : (C, T)
    Output : (C, 120, S)  where S = T - 3*(kernel_size-1)
    """
    def __init__(self, input_channels, kernel_size=5):
        super().__init__()
        self.ncf = 120
        C = input_channels
        self.cv1  = nn.Conv1d(C, C,            kernel_size, groups=C, padding='valid')
        self.cv2  = nn.Conv1d(C, C,            kernel_size, groups=C, padding='valid')
        self.cv3  = nn.Conv1d(C, self.ncf * C, kernel_size, groups=C, padding='valid')
        self.relu = nn.ReLU()

    def forward(self, x):           # (C, T)
        x = self.relu(self.cv1(x))
        x = self.relu(self.cv2(x))
        x = self.relu(self.cv3(x))  # (ncf*C, S)
        S = x.shape[1]
        C = x.shape[0] // self.ncf
        return x.reshape(C, self.ncf, S)   # (C, D=120, S)


class RTM(nn.Module):
    def __init__(self, odcm_out_shape, num_blocks, num_heads):
        super().__init__()
        C, D, S = odcm_out_shape
        self.M  = D
        self.weight = nn.Parameter(torch.randn(D, D))
        self.bias   = nn.Parameter(torch.zeros(S, C + 1, D))
        self.cls    = nn.Parameter(torch.zeros(S, 1, D))
        trunc_normal(self.bias, std=.02)
        trunc_normal(self.cls,  std=.02)
        self.tfbs = nn.ModuleList([GenericTFB(D, num_heads) for _ in range(num_blocks)])

    def forward(self, x):
        x = x.permute(2, 0, 1)                              # (S, C, D)
        z = torch.einsum('lm,ijm->ijl', self.weight, x)     # (S, C, D)
        z = torch.cat([self.cls, z], dim=1)                  # (S, C+1, D)
        z = z + self.bias
        for tfb in self.tfbs:
            z = tfb(x, z)
        return z


class STM(nn.Module):
    def __init__(self, rtm_out_shape, num_blocks, num_heads):
        super().__init__()
        S, Cp1, D = rtm_out_shape
        self.M  = D
        self.weight = nn.Parameter(torch.randn(D, D))
        self.bias   = nn.Parameter(torch.zeros(Cp1, S + 1, D))
        self.cls    = nn.Parameter(torch.zeros(Cp1, 1, D))
        trunc_normal(self.bias, std=.02)
        trunc_normal(self.cls,  std=.02)
        self.tfbs = nn.ModuleList([GenericTFB(D, num_heads) for _ in range(num_blocks)])

    def forward(self, x):
        x = x.transpose(0, 1)                               # (C+1, S, D)
        z = torch.einsum('lm,ijm->ijl', self.weight, x)     # (C+1, S, D)
        z = torch.cat([self.cls, z], dim=1)                  # (C+1, S+1, D)
        z = z + self.bias
        for tfb in self.tfbs:
            z = tfb(x, z)
        return z


class TTM(nn.Module):
    def __init__(self, stm_out_shape, num_submatrices, num_blocks, num_heads):
        super().__init__()
        Cp1, Sp1, D = stm_out_shape
        self.avgf = num_submatrices
        self.M    = num_submatrices
        assert D % num_submatrices == 0
        assert self.M % num_heads == 0
        flat = Cp1 * Sp1
        self.weight = nn.Parameter(torch.randn(self.M, flat))
        self.bias   = nn.Parameter(torch.zeros(num_submatrices + 1, self.M))
        self.cls    = nn.Parameter(torch.zeros(1, self.M))
        trunc_normal(self.bias, std=.02)
        trunc_normal(self.cls,  std=.02)
        self.tfbs = nn.ModuleList([
            TemporalTFB(self.M, num_heads, num_submatrices) for _ in range(num_blocks)
        ])
        self.ln = nn.LayerNorm(self.M)

    def forward(self, x):
        Cp1, Sp1, D = x.shape
        seg  = D // self.avgf
        flat = Cp1 * Sp1
        xc   = x.permute(2, 0, 1).reshape(self.avgf, seg, Cp1, Sp1).mean(dim=1)
        alt  = xc.reshape(self.avgf, flat)
        z    = torch.einsum('lm,im->il', self.weight, alt)
        z    = torch.cat([self.cls, z], dim=0)
        z    = z + self.bias
        for tfb in self.tfbs:
            z = tfb(x, z)
        z   = self.ln(z)
        out = torch.einsum('tm,mf->tf', z, self.weight)
        return out.reshape(self.avgf + 1, Cp1, Sp1)


class CNNDecoder(nn.Module):
    def __init__(self, ttm_out_shape, num_classes, CF_second=2):
        super().__init__()
        Mp1, Cp1, Sp1 = ttm_out_shape
        self.cvd1 = nn.Conv1d(Cp1,  1,         kernel_size=1)
        self.cvd2 = nn.Conv1d(Sp1,  CF_second,  kernel_size=1)
        self.cvd3 = nn.Conv1d(Mp1,  Mp1 // 2,   kernel_size=1)
        self.fc   = nn.Linear((Mp1 // 2) * CF_second, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.permute(2, 1, 0)
        x = self.relu(self.cvd1(x)).squeeze(1)
        x = self.relu(self.cvd2(x)).transpose(0, 1)
        x = self.relu(self.cvd3(x))
        x = self.fc(x.reshape(1, -1))
        return x


# ─── L_SCLNet with EEGformer backbone ────────────────────────────────────
class L_SCLNet_EEGformer(nn.Module):
    """
    L_SCLNet with EEGformer backbone for the Epileptic Seizure Recognition CSV.

    Key parameters vs TUH version:
        input_channels = 1   (single pseudo-channel)
        time_steps     = 177 (feature vector length)
        kernel_size    = 5   (reduced — 177 is much shorter than 512)

    Shape trace (C=1, kernel_size=5, time_steps=177):
        ODCM : S = 177 - 3*(5-1) = 165
        RTM  : (165, 2, 120)
        STM  : (2, 166, 120)
        TTM  : (7, 2, 166)  with M=6
    """
    def __init__(
        self,
        input_channels  = 1,
        time_steps      = 177,
        num_classes     = 2,
        kernel_size     = 5,    # ← reduced for 177-sample input
        num_blocks      = 3,
        num_heads_rtm   = 6,
        num_heads_stm   = 6,
        num_heads_ttm   = 6,
        num_submatrices = 6,
        CF_second       = 2,
    ):
        super().__init__()
        ncf = 120
        C   = input_channels
        D   = ncf
        S   = time_steps - 3 * (kernel_size - 1)

        assert D % num_heads_rtm == 0
        assert D % num_heads_stm == 0
        assert num_submatrices % num_heads_ttm == 0
        assert D % num_submatrices == 0

        self.odcm    = ODCM(C, kernel_size)
        self.rtm     = RTM((C, D, S),       num_blocks, num_heads_rtm)
        self.stm     = STM((S, C+1, D),     num_blocks, num_heads_stm)
        self.ttm     = TTM((C+1, S+1, D),   num_submatrices, num_blocks, num_heads_ttm)
        self.decoder = CNNDecoder((num_submatrices+1, C+1, S+1), num_classes, CF_second)

    def forward(self, x):
        # x : (B, C, T)
        outs = []
        for i in range(x.shape[0]):
            xi = self.odcm(x[i])
            xi = self.rtm(xi)
            xi = self.stm(xi)
            xi = self.ttm(xi)
            xi = self.decoder(xi)
            outs.append(xi)
        return torch.cat(outs, dim=0)   # (B, num_classes)

print('Model definition complete.')

Model definition complete.


## 5. Training

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

model = L_SCLNet_EEGformer(
    input_channels  = 1,
    time_steps      = 177,
    num_classes     = 2,
    kernel_size     = 5,
    num_blocks      = 3,
    num_heads_rtm   = 6,
    num_heads_stm   = 6,
    num_heads_ttm   = 6,
    num_submatrices = 6,
    CF_second       = 2,
).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

# Class-weighted loss
counts    = np.bincount(y[train_dataset.indices].numpy())
class_wts = torch.tensor(counts.sum() / (2 * counts), dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_wts)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

NUM_EPOCHS = 30
scheduler  = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-4,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1,
)

best_val_acc    = 0.0
best_model_path = 'best_lsclnet_epileptic_csv.pth'
history         = {'train_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}
LOG_EPOCHS      = set(range(1, NUM_EPOCHS + 1, 10)) | {NUM_EPOCHS}

for epoch in range(1, NUM_EPOCHS + 1):
    # ── train ──────────────────────────────────────────────────────────────
    model.train()
    total_loss, all_preds, all_labels_list = 0.0, [], []
    for xb, yb in tqdm(train_loader, desc=f'Epoch {epoch:02d}', leave=False):
        xb, yb = xb.to(device), yb.to(device)
        logits  = model(xb)
        loss    = criterion(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        all_preds.extend(torch.argmax(logits, 1).cpu().numpy())
        all_labels_list.extend(yb.cpu().numpy())

    train_acc = accuracy_score(all_labels_list, all_preds)
    avg_loss  = total_loss / len(train_loader)
    history['train_loss'].append(avg_loss)
    history['train_acc'].append(train_acc)

    # ── validate ──────────────────────────────────────────────────────────
    model.eval()
    val_preds, val_labels_list = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            val_preds.extend(torch.argmax(model(xb), 1).cpu().numpy())
            val_labels_list.extend(yb.numpy())

    val_acc = accuracy_score(val_labels_list, val_preds)
    val_f1  = f1_score(val_labels_list, val_preds, zero_division=0)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    if epoch in LOG_EPOCHS:
        print(f'[Epoch {epoch:02d}] loss={avg_loss:.4f} | '
              f'train_acc={train_acc*100:.2f}% | '
              f'val_acc={val_acc*100:.2f}% | '
              f'val_f1={val_f1:.4f} | '
              f'lr={scheduler.get_last_lr()[0]:.2e}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        if epoch in LOG_EPOCHS:
            print(f'  ✅ Best model saved (val_acc={val_acc*100:.2f}%)')

print(f'\n🏆  Best val acc: {best_val_acc*100:.2f}%')

Using device: cuda
Parameters: 1,176,177


[Epoch 01] loss=0.6312 | train_acc=52.14% | val_acc=23.18% | val_f1=0.3601 | lr=8.48e-05


[Epoch 11] loss=0.4867 | train_acc=50.78% | val_acc=20.35% | val_f1=0.3382 | lr=2.39e-04


[Epoch 21] loss=0.1874 | train_acc=88.67% | val_acc=91.44% | val_f1=0.8937 | lr=7.47e-05


[Epoch 30] loss=0.0423 | train_acc=98.81% | val_acc=99.35% | val_f1=0.9921 | lr=1.55e-09

🏆  Best val acc: 99.35%


## 6. Test Evaluation

In [8]:
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

test_preds, test_labels_list = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        test_preds.extend(torch.argmax(model(xb), 1).cpu().numpy())
        test_labels_list.extend(yb.numpy())

print('📊  Test Set Results')
print(f'   Accuracy  : {accuracy_score(test_labels_list, test_preds)*100:.2f}%')
print(f'   F1 Score  : {f1_score(test_labels_list, test_preds):.4f}')
print(f'   Precision : {precision_score(test_labels_list, test_preds):.4f}')
print(f'   Recall    : {recall_score(test_labels_list, test_preds):.4f}')
print()
print('Confusion Matrix:')
print(confusion_matrix(test_labels_list, test_preds))

📊  Test Set Results
   Accuracy  : 99.35%
   F1 Score  : 0.9921
   Precision : 0.9887
   Recall    : 0.9956

Confusion Matrix:
[[1849   13]
 [   3  435]]
